# FREUID Overlay Detector — Colab Training

Setup-only notebook. All logic lives in `src/`. This just wires up the environment, data, and launches training.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Clone repo

In [ ]:
import os

REPO_URL = ""  # <-- paste your repo URL here, e.g. "https://github.com/user/freuid-challenge.git"
BRANCH = "photo-overlay"
WORK_DIR = "/content/freuid-challenge"

if REPO_URL:
    if not os.path.exists(WORK_DIR):
        !git clone -b {BRANCH} {REPO_URL} {WORK_DIR}
    os.chdir(WORK_DIR)
else:
    # Manual upload: put the repo files in /content/freuid-challenge/
    assert os.path.exists(WORK_DIR), f"Upload repo to {WORK_DIR} or set REPO_URL"
    os.chdir(WORK_DIR)

print(f"Working dir: {os.getcwd()}")

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q facenet-pytorch --no-deps

## 4. Download dataset

In [ ]:
!pip install -q kagglehub

import kagglehub
import os

DATA_DIR = "data/raw"

if not os.path.exists(f"{DATA_DIR}/train_labels.csv"):
    path = kagglehub.competition_download('the-freuid-challenge-2026-ijcai-ecai')
    print("Path to competition files:", path)
    os.makedirs(DATA_DIR, exist_ok=True)
    !cp -r {path}/* {DATA_DIR}/
    print("Dataset copied to data/raw/")
else:
    print("Dataset already present.")

!ls {DATA_DIR}/

## 5. Write Colab config

Same as `overlay_baseline.yaml` but with debug off, more workers, and larger batch size for GPU.

In [ ]:
%%writefile configs/overlay_colab.yaml
seed: 42

debug:
  enabled: false

data:
  train_csv: data/raw/train_labels.csv
  train_img_dir: data/raw/train/train
  test_img_dir: data/raw/public_test/public_test
  val_doc_type: MAURITIUS/ID
  img_size: 224
  num_workers: 2
  crop_margin: 0.75
  crop_cache_dir: data/processed/overlay_crops

model:
  type: overlay
  noise_frontend: bayar
  noise_feat_dim: 128
  use_rgb_stream: false
  rgb_backbone: resnet34
  rgb_pretrained: true
  fusion_dim: 128

train:
  epochs: 10
  batch_size: 64
  lr: 1.0e-4
  weight_decay: 1.0e-4
  scheduler: cosine

output:
  model_dir: outputs/models
  pred_dir: outputs/predictions
  sub_dir: outputs/submissions
  experiment_name: overlay_colab

## 6. Train

In [ ]:
!python -m src.train --config configs/overlay_colab.yaml

## 7. Predict + submission

In [ ]:
!python -m src.predict --config configs/overlay_colab.yaml
!python -m src.make_submission --config configs/overlay_colab.yaml

## 8. Download results

In [ ]:
from google.colab import files
files.download("outputs/submissions/overlay_colab.csv")
files.download("outputs/models/overlay_colab_best.pt")